# H2/STO-3G VQE with Qiskit Nature

Goal: build the complete electronic-structure-to-qubit pipeline for H2 in STO-3G.

Pipeline:

1. Define molecular geometry and basis.
2. Use PySCF through Qiskit Nature to obtain the electronic-structure problem.
3. Build the second-quantized fermionic Hamiltonian.
4. Map the fermionic Hamiltonian to qubits using Jordan-Wigner.
5. Build a Hartree-Fock initial state and UCCSD ansatz.
6. Run VQE with a classical optimizer.
7. Compare against an exact classical eigensolver reference.


In [1]:
import importlib.metadata as metadata

packages = [
    'qiskit',
    'qiskit-nature',
    'qiskit-algorithms',
    'pyscf',
    'numpy',
]

for package in packages:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED')


qiskit: 1.4.5
qiskit-nature: 0.7.2
qiskit-algorithms: 0.3.1
pyscf: 2.13.0
numpy: 2.4.4


## 1. Define molecule, basis, charge, and spin

We start with neutral singlet H2 at a fixed bond distance. This is the smallest useful electronic-structure problem because STO-3G H2 has two spatial orbitals, four spin orbitals, and therefore four Jordan-Wigner qubits before any symmetry reduction.

In [2]:
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit

bond_length_angstrom = 0.735
basis = 'sto3g'

driver = PySCFDriver(
    atom=f'H 0 0 0; H 0 0 {bond_length_angstrom}',
    unit=DistanceUnit.ANGSTROM,
    charge=0,
    spin=0,
    basis=basis,
)

problem = driver.run()

print('Molecule: H2')
print(f'Bond length: {bond_length_angstrom} angstrom')
print(f'Basis: {basis}')
print(f'Charge: {problem.molecule.charge}')
print(f'Spin multiplicity parameter, 2S: {problem.molecule.multiplicity - 1}')
print(f'Number of spatial orbitals: {problem.num_spatial_orbitals}')
print(f'Number of spin orbitals: {2 * problem.num_spatial_orbitals}')
print(f'Number of particles, alpha/beta: {problem.num_particles}')


Molecule: H2
Bond length: 0.735 angstrom
Basis: sto3g
Charge: 0
Spin multiplicity parameter, 2S: 0
Number of spatial orbitals: 2
Number of spin orbitals: 4
Number of particles, alpha/beta: (1, 1)


## 2. Build the fermionic Hamiltonian

Qiskit Nature represents the electronic Hamiltonian in second quantization. Conceptually, this is the molecular electronic Hamiltonian expressed in creation and annihilation operators over spin orbitals.

In [3]:
fermionic_op = problem.hamiltonian.second_q_op()

print(type(fermionic_op))
print(f'Number of fermionic terms: {len(fermionic_op)}')
print()
print(fermionic_op)


<class 'qiskit_nature.second_q.operators.fermionic_op.FermionicOp'>
Number of fermionic terms: 36

Fermionic Operator
number spin orbitals=4, number terms=36
  0.33785507740175813 * ( +_0 +_0 -_0 -_0 )
+ 0.3322908651276482 * ( +_0 +_1 -_1 -_0 )
+ 0.33785507740175813 * ( +_0 +_2 -_2 -_0 )
+ 0.3322908651276482 * ( +_0 +_3 -_3 -_0 )
+ 0.09046559989211572 * ( +_0 +_0 -_1 -_1 )
+ 0.09046559989211572 * ( +_0 +_1 -_0 -_1 )
+ 0.09046559989211572 * ( +_0 +_2 -_3 -_1 )
+ 0.09046559989211572 * ( +_0 +_3 -_2 -_1 )
+ 0.09046559989211572 * ( +_1 +_0 -_1 -_0 )
+ 0.09046559989211572 * ( +_1 +_1 -_0 -_0 )
+ 0.09046559989211572 * ( +_1 +_2 -_3 -_0 )
+ 0.09046559989211572 * ( +_1 +_3 -_2 -_0 )
+ 0.3322908651276482 * ( +_1 +_0 -_0 -_1 )
+ 0.34928686136600884 * ( +_1 +_1 -_1 -_1 )
+ 0.3322908651276482 * ( +_1 +_2 -_2 -_1 )
+ 0.34928686136600884 * ( +_1 +_3 -_3 -_1 )
+ 0.33785507740175813 * ( +_2 +_0 -_0 -_2 )
+ 0.3322908651276482 * ( +_2 +_1 -_1 -_2 )
+ 0.33785507740175813 * ( +_2 +_2 -_2 -_2 )
+ 0.3322908

## 3. Map fermions to qubits using Jordan-Wigner

Jordan-Wigner gives the most direct conceptual mapping: each spin orbital occupation maps to one qubit. For H2/STO-3G, four spin orbitals become four qubits.

In [4]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()
qubit_op = mapper.map(fermionic_op)

print(type(qubit_op))
print(f'Number of qubits: {qubit_op.num_qubits}')
print(f'Number of Pauli terms: {len(qubit_op)}')
print()
print(qubit_op)


<class 'qiskit.quantum_info.operators.symplectic.sparse_pauli_op.SparsePauliOp'>
Number of qubits: 4
Number of Pauli terms: 15

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.81054798+0.j,  0.17218393+0.j, -0.22575349+0.j,  0.12091263+0.j,
  0.17218393+0.j,  0.16892754+0.j, -0.22575349+0.j,  0.16614543+0.j,
  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,  0.0452328 +0.j,
  0.16614543+0.j,  0.17464343+0.j,  0.12091263+0.j])


## 4. Classical exact reference

For this toy system, we can diagonalize the qubit Hamiltonian exactly. This is not scalable, but it gives a reference energy for checking VQE.

In [5]:
import numpy as np

# Exact classical reference by direct matrix diagonalization.
# This is possible only because H2/STO-3G has four qubits.
hamiltonian_matrix = qubit_op.to_matrix()
electronic_eigenvalues = np.linalg.eigvalsh(hamiltonian_matrix)

exact_electronic_energy = float(np.min(electronic_eigenvalues).real)
exact_energy = exact_electronic_energy + float(problem.nuclear_repulsion_energy)

print(f'Exact electronic energy: {exact_electronic_energy:.12f} Hartree')
print(f'Nuclear repulsion energy: {problem.nuclear_repulsion_energy:.12f} Hartree')
print(f'Exact total ground-state energy: {exact_energy:.12f} Hartree')


Exact electronic energy: -1.857275030202 Hartree
Nuclear repulsion energy: 0.719968994449 Hartree
Exact total ground-state energy: -1.137306035753 Hartree


## 5. Build Hartree-Fock initial state and UCCSD ansatz

This is the direct connection to classical quantum chemistry. Classical CCSD applies an exponential cluster operator to a reference determinant. Quantum UCCSD uses a unitary form of the cluster operator, usually starting from a Hartree-Fock reference state.

In [6]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

initial_state = HartreeFock(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    qubit_mapper=mapper,
)

ansatz = UCCSD(
    num_spatial_orbitals=problem.num_spatial_orbitals,
    num_particles=problem.num_particles,
    qubit_mapper=mapper,
    initial_state=initial_state,
)

print(f'Ansatz: {ansatz.__class__.__name__}')
print(f'Number of qubits in ansatz: {ansatz.num_qubits}')
print(f'Number of variational parameters: {ansatz.num_parameters}')
print(f'Circuit depth: {ansatz.decompose().depth()}')

ansatz.decompose().draw('text')


Ansatz: UCCSD
Number of qubits in ansatz: 4
Number of variational parameters: 3
Circuit depth: 4


┌───┐┌───────────────────────────────┐┌───────────────────────────────┐»
q_0: ┤ X ├┤0                              ├┤0                              ├»
     └───┘│                               ││                               │»
q_1: ─────┤1                              ├┤1                              ├»
     ┌───┐│  exp(-it (IIXY + IIYX))(t[0]) ││  exp(-it (XYII + YXII))(t[1]) │»
q_2: ┤ X ├┤2                              ├┤2                              ├»
     └───┘│                               ││                               │»
q_3: ─────┤3                              ├┤3                              ├»
          └───────────────────────────────┘└───────────────────────────────┘»
«     ┌─────────────────────────────────────────────────────────────────────────┐
«q_0: ┤0                                                                        ├
«     │                                                                         │
«q_1: ┤1                                                                        ├
«     │  exp(-it (YYXY + XYYY + XXXY + YXYY + XYXX + YYYX + YXXX + XXYX))(t[2]) │
«q_2: ┤2                                                                        ├
«     │                                                                         │
«q_3: ┤3                                                                        ├
«     └─────────────────────────────────────────────────────────────────────────┘

## 6. Run VQE

VQE minimizes the expectation value of the qubit Hamiltonian with respect to ansatz parameters. Here we use a statevector estimator, so this is an ideal noiseless simulation, not a shot-based hardware run.

In [7]:
import numpy as np
from scipy.optimize import minimize
from qiskit.quantum_info import Statevector

history = []

def energy_from_parameters(parameters):
    bound_circuit = ansatz.assign_parameters(parameters, inplace=False)
    state = Statevector.from_instruction(bound_circuit)
    electronic_energy = np.real(state.expectation_value(qubit_op))
    total_energy = electronic_energy + float(problem.nuclear_repulsion_energy)
    history.append(float(total_energy))
    return float(total_energy)

initial_point = np.zeros(ansatz.num_parameters)

opt_result = minimize(
    energy_from_parameters,
    initial_point,
    method='SLSQP',
    options={'maxiter': 1000, 'ftol': 1e-12},
)

vqe_energy = float(opt_result.fun)
error_hartree = vqe_energy - exact_energy

print(f'Optimizer success: {opt_result.success}')
print(f'Optimizer message: {opt_result.message}')
print(f'VQE total ground-state energy: {vqe_energy:.12f} Hartree')
print(f'Exact total ground-state energy: {exact_energy:.12f} Hartree')
print(f'Error: {error_hartree:.12e} Hartree')
print(f'Number of objective evaluations: {len(history)}')


Optimizer success: True
Optimizer message: Optimization terminated successfully
VQE total ground-state energy: -1.137306035753 Hartree
Exact total ground-state energy: -1.137306035753 Hartree
Error: 3.952393967666e-14 Hartree
Number of objective evaluations: 17


## 7. Summary

This cell collects the required milestone outputs.

In [8]:
summary = {
    'molecule': 'H2',
    'bond_length_angstrom': bond_length_angstrom,
    'basis': basis,
    'charge': problem.molecule.charge,
    'spin_2S': problem.molecule.multiplicity - 1,
    'spatial_orbitals': problem.num_spatial_orbitals,
    'spin_orbitals': 2 * problem.num_spatial_orbitals,
    'particles_alpha_beta': problem.num_particles,
    'mapping': mapper.__class__.__name__,
    'qubits': qubit_op.num_qubits,
    'pauli_terms': len(qubit_op),
    'ansatz': ansatz.__class__.__name__,
    'ansatz_parameters': ansatz.num_parameters,
    'optimizer': 'scipy.optimize.minimize, SLSQP',
    'classical_reference_energy_hartree': exact_energy,
    'vqe_energy_hartree': vqe_energy,
    'error_hartree': error_hartree,
}

for key, value in summary.items():
    print(f'{key}: {value}')


molecule: H2
bond_length_angstrom: 0.735
basis: sto3g
charge: 0
spin_2S: 0
spatial_orbitals: 2
spin_orbitals: 4
particles_alpha_beta: (1, 1)
mapping: JordanWignerMapper
qubits: 4
pauli_terms: 15
ansatz: UCCSD
ansatz_parameters: 3
optimizer: scipy.optimize.minimize, SLSQP
classical_reference_energy_hartree: -1.1373060357534
vqe_energy_hartree: -1.1373060357533604
error_hartree: 3.952393967665557e-14


## Limitations

1. This is an ideal statevector simulation, not a hardware calculation.
2. No sampling noise is included.
3. No device noise, decoherence, readout error, or transpilation constraints are included.
4. H2/STO-3G is too small to demonstrate quantum advantage.
5. Exact diagonalization is possible here only because the Hilbert space is tiny.
6. Jordan-Wigner is conceptually simple but not always the most resource-efficient mapping.
7. UCCSD is chemically motivated, but circuit depth becomes a serious issue for larger active spaces.